# NeuroQ: demonstrația, etapă cu etapă

Reconstruim cele două copii ale unei regiuni din cromozom (phasing) din citiri de ADN, cu QAOA pe simulator. Fiecare celulă de mai jos arată o etapă a pipeline-ului. Instanța implicită este exemplul din laboratorul organizatorilor (5 SNP-uri, 9 citiri), care rulează în câteva secunde. Poți schimba `INSTANCE` (`notebook`, `illumina`, `nanopore`), `ENCODING` (`pozitii` sau `citiri`) și `FIX`.

In [ ]:
import sys
from pathlib import Path

# găsim rădăcina repo-ului (folderul care conține pachetul neuroq)
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'neuroq').is_dir())
sys.path.insert(0, str(ROOT))

%matplotlib inline
import matplotlib.pyplot as plt
from IPython.display import display

from neuroq import viz
from neuroq.experiments.stages import load_case, resolve_encoding, run_stages

INSTANCE = 'notebook'
ENCODING = None  # None = codificarea implicită a instanței
FIX = False

def show(fig):
    """Afișează figura o singură dată și o închide."""
    if fig is not None:
        display(fig)
        plt.close(fig)

## 1. Datele

Fiecare rând este o citire, fiecare coloană un SNP. 0 înseamnă litera de referință, 1 litera alternativă, iar celulele goale sunt poziții pe care citirea nu le acoperă. Nu știm din ce copie vine fiecare citire.

In [ ]:
case = load_case(INSTANCE)
m = case.matrix
show(viz.plot_reads_matrix(m.values, m.read_ids, m.snp_ids))

## 2. Rulăm pipeline-ul

`run_stages` face tot calculul (codificare, referința exactă, antrenarea și eșantionarea QAOA, decodarea, metricile) și nu desenează nimic. Celulele următoare doar afișează ce a calculat.

In [ ]:
kind = resolve_encoding(INSTANCE, ENCODING)
result = run_stages(case, kind, FIX)
print(f'codificarea pe {kind}: {result.problem.n_qubits} variabile; '
      f'QAOA pe {result.solved.n_qubits} qubiți')
if result.other_encoding:
    print('cealaltă codificare (nu se rulează):', result.other_encoding)

## 3. Voturile a și d

Pentru fiecare pereche de variabile numărăm de câte ori citirile spun „la fel” (a) și de câte ori spun „opus” (d).

In [ ]:
show(viz.plot_evidence(result.a, result.d, result.labels))

## 4. Hamiltonianul Ising

Cuplajul fiecărei perechi este J = (d − a) / 2: J pozitiv respinge (variabilele vor valori diferite), J negativ atrage (vor valori egale). Constanta din titlu face ca energia să fie exact numărul de voturi încălcate.

In [ ]:
show(viz.plot_ising(result.problem.J, result.problem.offset, result.labels))

## 5. Graful

Nodurile sunt variabilele (qubiții), muchiile sunt cuplajele: albastru continuu atrage, roșu întrerupt respinge, grosimea arată mărimea.

In [ ]:
show(viz.plot_graph(result.problem.J, result.labels))

## 6. Circuitul QAOA

Un strat (p = 1): H pe toți qubiții, câte o poartă RZZ pentru fiecare cuplaj nenul, apoi RX pe fiecare qubit, cu unghiurile antrenate.

In [ ]:
show(viz.plot_circuit(result.circuit))

## 7. Antrenarea

COBYLA caută unghiurile care scad energia medie ⟨H⟩, pornind din două puncte cu seed fix. Linia punctată este optimul exact.

In [ ]:
show(viz.plot_qaoa_cost(result.cost_history, result.exact.energy, result.best_start))

## 8. Cele mai probabile 10 șiruri

După antrenare măsurăm circuitul de 2048 de ori. Barele verzi sunt șiruri cu energia optimă; fără fixarea unui qubit, soluția și oglinda ei apar amândouă.

In [ ]:
show(viz.plot_top_samples(result.top_samples, result.exact.energy))
show(viz.plot_graph(result.problem.J, result.labels, bitstring=result.full_bitstring,
                    energy=result.problem.energy(result.full_bitstring),
                    title='Soluția QAOA pe graf'))

## 9. Rezultatul

Decodăm șirul ales în cele două copii și le comparăm cu adevărul (oglinda contează ca potrivire).

In [ ]:
if m.true_haplotypes is not None:
    show(viz.plot_haplotypes(result.haplotypes, m.true_haplotypes))
{k: round(v, 3) for k, v in result.metrics.items()}